In [14]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [15]:
!python -m pip install --upgrade pip

zsh:1: command not found: python


In [16]:
pip install streamlit

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [17]:
import streamlit as st
import datetime

# 1. Seiteneinstellungen optimieren
st.set_page_config(page_title="VibeCleaning", page_icon="🧹", layout="centered")

# Wochentage als feste Liste für das Tracking
wochentage = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]

# 2. Datenstruktur im st.session_state initialisieren
# Das sorgt dafür, dass die App die Klicks speichert, solange der Server läuft.
if "tracking_data" not in st.session_state:
    # Struktur: { KW: { aufgabe_index: { "Mo": True/False, ... } } }
    st.session_state.tracking_data = {}

# 3. Festgelegtes WG-Setup
wg_crew = ["Nico", "Kiki", "Bruno"]
aufgaben = ["Bad putzen 🧼", "Küche & Böden 🍳", "Müll & Altglas 🗑️"]

anzahl_mitglieder = len(wg_crew)
aktuell_kw = datetime.datetime.now().isocalendar()[1]
letzte_kw = aktuell_kw - 1

# Sicherstellen, dass für die aktuelle und letzte Woche Datenstrukturen existieren
for kw in [aktuell_kw, letzte_kw]:
    if kw not in st.session_state.tracking_data:
        st.session_state.tracking_data[kw] = {
            i: {tag: False for tag in wochentage} for i in range(len(aufgaben))
        }

# --- UI LAYOUT ---

st.title("🧹 VibeCleaning")

# FEATURE 1: Aktiver Nutzer (Dropdown in der Seitenleiste)
st.sidebar.header("👤 Profil")
aktiver_nutzer = st.sidebar.selectbox("Wer nutzt die App gerade?", wg_crew)
st.sidebar.write(f"Hallo **{aktiver_nutzer}**! Viel Spaß beim Putzen. 🙌")

# Kalenderwochen-Info
st.info(f"📅 **Aktuelle Kalenderwoche:** {aktuell_kw}")

# Haupt-Tabs für die Übersicht
tab1, tab2 = st.tabs(["📌 Aktuelle Woche", "📜 Historie (Letzte Woche)"])

# ----------------------------------------------------
# TAB 1: AKTUELLE WOCHE (FEATURE 2: Toggle-Buttons)
# ----------------------------------------------------
with tab1:
    st.subheader("Deine Aufgaben für diese Woche:")
    
    for i, aufgabe in enumerate(aufgaben):
        # Rotationslogik basierend auf der aktuellen KW
        bewohner_index = (i + aktuell_kw) % anzahl_mitglieder
        zustandiger = wg_crew[bewohner_index]
        
        # Visuelle Box für die Aufgabe
        with st.container():
            st.markdown(f"### {aufgabe}")
            st.markdown(f"👤 **Zuständig:** `{zustandiger}`")
            
            # 7 Buttons nebeneinander für die Wochentage
            cols = st.columns(7)
            for j, tag in enumerate(wochentage):
                with cols[j]:
                    # Aktuellen Zustand aus dem Speicher holen
                    ist_erledigt = st.session_state.tracking_data[aktuell_kw][i][tag]
                    
                    # Visuelles Feedback: Grüner Haken wenn erledigt, sonst normaler Buchstabe
                    button_label = f"✅ {tag}" if ist_erledigt else tag
                    
                    # Toggle-Logik bei Klick
                    if st.button(button_label, key=f"btn_akt_{i}_{tag}", use_container_width=True):
                        # Zustand umdrehen (True -> False / False -> True)
                        st.session_state.tracking_data[aktuell_kw][i][tag] = not ist_erledigt
                        st.rerun() # Seite neu laden, um das visuelle Feedback sofort anzuzeigen
            st.markdown("---")

# ----------------------------------------------------
# TAB 2: HISTORIE (FEATURE 3: Letzte Woche anzeigen)
# ----------------------------------------------------
with tab2:
    st.subheader(f"Ergebnisse aus der Vorwoche (KW {letzte_kw})")
    
    # Ausklappbarer Bereich (Expander) für die Historie
    with st.expander("📊 Detailübersicht öffnen", expanded=True):
        for i, aufgabe in enumerate(aufgaben):
            # Rotationslogik für die LETZTE Woche berechnen
            bewohner_index_letzte = (i + letzte_kw) % anzahl_mitglieder
            zustandiger_letzte = wg_crew[bewohner_index_letzte]
            
            st.markdown(f"**{aufgabe}** (Verantwortlich: *{zustandiger_letzte}*)")
            
            # Zeige an, an welchen Tagen die Aufgabe erledigt wurde
            erledigte_tage = []
            for tag in wochentage:
                if st.session_state.tracking_data[letzte_kw][i][tag]:
                    erledigte_tage.append(f"🟢 {tag}")
                else:
                    erledigte_tage.append(f"⚪ {tag}")
            
            # Darstellung als Reihe von Status-Punkten
            st.write(" ".join(erledigte_tage))
            st.markdown("")

2026-05-29 22:55:09.112 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 22:55:09.113 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 22:55:09.114 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 22:55:09.114 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 22:55:09.115 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 22:55:09.115 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 22:55:09.115 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-29 22:55:09.116 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [18]:
%%writefile app.py
import streamlit as st
import datetime

st.set_page_config(page_title="VibeCleaning", page_icon="🧹", layout="centered")

wochentage = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]

if "tracking_data" not in st.session_state:
    st.session_state.tracking_data = {}

wg_crew = ["Nico", "Kiki", "Bruno"]
aufgaben = ["Bad putzen 🧼", "Küche & Böden 🍳", "Müll & Altglas 🗑️"]

anzahl_mitglieder = len(wg_crew)
aktuell_kw = datetime.datetime.now().isocalendar()[1]
letzte_kw = aktuell_kw - 1

for kw in [aktuell_kw, letzte_kw]:
    if kw not in st.session_state.tracking_data:
        st.session_state.tracking_data[kw] = {
            i: {tag: False for tag in wochentage} for i in range(len(aufgaben))
        }

st.title("🧹 VibeCleaning")

st.sidebar.header("👤 Profil")
aktiver_nutzer = st.sidebar.selectbox("Wer nutzt die App gerade?", wg_crew)
st.sidebar.write(f"Hallo **{aktiver_nutzer}**! Viel Spaß beim Putzen. 🙌")

st.info(f"📅 **Aktuelle Kalenderwoche:** {aktuell_kw}")

tab1, tab2 = st.tabs(["📌 Aktuelle Woche", "📜 Historie (Letzte Woche)"])

with tab1:
    st.subheader("Deine Aufgaben für diese Woche:")
    for i, aufgabe in enumerate(aufgaben):
        bewohner_index = (i + aktuell_kw) % anzahl_mitglieder
        zustandiger = wg_crew[bewohner_index]
        
        with st.container():
            st.markdown(f"### {aufgabe}")
            st.markdown(f"👤 **Zuständig:** `{zustandiger}`")
            
            cols = st.columns(7)
            for j, tag in enumerate(wochentage):
                with cols[j]:
                    ist_erledigt = st.session_state.tracking_data[aktuell_kw][i][tag]
                    button_label = f"✅ {tag}" if ist_erledigt else tag
                    if st.button(button_label, key=f"btn_akt_{i}_{tag}", use_container_width=True):
                        st.session_state.tracking_data[aktuell_kw][i][tag] = not ist_erledigt
                        st.rerun()
            st.markdown("---")

with tab2:
    st.subheader(f"Ergebnisse aus der Vorwoche (KW {letzte_kw})")
    with st.expander("📊 Detailübersicht öffnen", expanded=True):
        for i, aufgabe in enumerate(aufgaben):
            bewohner_index_letzte = (i + letzte_kw) % anzahl_mitglieder
            zustandiger_letzte = wg_crew[bewohner_index_letzte]
            
            st.markdown(f"**{aufgabe}** (Verantwortlich: *{zustandiger_letzte}*)")
            erledigte_tage = []
            for tag in wochentage:
                if st.session_state.tracking_data[letzte_kw][i][tag]:
                    erledigte_tage.append(f"🟢 {tag}")
                else:
                    erledigte_tage.append(f"⚪ {tag}")
            st.write(" ".join(erledigte_tage))
            st.markdown("")

Writing app.py
